# 04 — Model Optimization
**Applying 5 optimization techniques to the winning model**

Each technique is applied independently and measured against the baseline.
At the end we combine the best-performing ones into a final optimized model.

Techniques covered:
1. Gradient Checkpointing — reduce VRAM at cost of ~20% more compute
2. Knowledge Distillation — compress into smaller student model
3. INT8 Quantization — 4x inference memory reduction
4. Optuna HPO — find smallest architecture hitting R² target
5. Combined — best of all above merged

> **Prerequisite:** `03_comparison.ipynb` must have run — `config.json` must have `winner_model`.

## 0. Setup & Load Winner

In [ ]:
import sys, json, time, warnings
sys.path.insert(0, '.')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler

from utils.metrics import compute_metrics, ResourceTracker, print_metrics

with open('./config.json') as f:
    CONFIG = json.load(f)

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = CONFIG['use_amp'] and torch.cuda.is_available()
WINNER  = CONFIG.get('winner_model', 'chemberta2')  # default to chemberta2
SEED    = CONFIG['random_seed']
torch.manual_seed(SEED)

df_train = pd.read_csv('./data/train.csv')
df_val   = pd.read_csv('./data/val.csv')
df_test  = pd.read_csv('./data/test.csv')

print(f'Winner model:   {WINNER}')
print(f'Device:         {DEVICE}')
print(f'AMP:            {USE_AMP}')
if torch.cuda.is_available():
    print(f'VRAM:           {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## Load Baseline Metrics
These are the numbers we need to beat (or match with fewer resources).

In [ ]:
import json

with open(f'./results/metrics_{WINNER}.json') as f:
    baseline_metrics = json.load(f)
with open(f'./results/resource_{WINNER}.json') as f:
    baseline_resources = json.load(f)

BASELINE = {**baseline_metrics, **baseline_resources}

print('Baseline (unoptimized) performance:')
print(f'  Test R²:       {BASELINE.get("test_r2", "N/A")}')
print(f'  Test RMSE:     {BASELINE.get("test_rmse", "N/A")}')
print(f'  Spearman ρ:    {BASELINE.get("test_spearman_rho", "N/A")}')
print(f'  Peak VRAM:     {BASELINE.get("peak_vram_mb", 0):.0f} MB')
print(f'  Train time:    {BASELINE.get("train_time_min", 0):.1f} min')
print(f'  CO2:           {BASELINE.get("kg_co2", 0)*1e6:.3f} mg')

## Helper — Shared Model & Data Loaders
Reused across all optimization experiments below.

In [ ]:
# Only run this if winner is chemberta2
# For other winners, adapt the model class accordingly

from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader

MODEL_NAME = 'seyonec/ChemBERTa-zinc-base-v1'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

class SMILESDataset(Dataset):
    def __init__(self, smiles_list, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            smiles_list, truncation=True, padding='max_length',
            max_length=max_length, return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'label':          self.labels[idx]
        }

class ChemBERTaRegressor(nn.Module):
    def __init__(self, model_name, hidden_size=256, dropout=0.2):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        bert_hidden = self.bert.config.hidden_size
        self.regressor = nn.Sequential(
            nn.Linear(bert_hidden, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.regressor(cls).squeeze(-1)

def make_loaders(batch_size=32):
    train_ds = SMILESDataset(df_train['Drug'].tolist(), df_train['Y_log'].values, tokenizer)
    val_ds   = SMILESDataset(df_val['Drug'].tolist(),   df_val['Y_log'].values,   tokenizer)
    test_ds  = SMILESDataset(df_test['Drug'].tolist(),  df_test['Y_log'].values,  tokenizer)
    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=4, pin_memory=True),
        DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True),
        DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True),
    )

def evaluate_model(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for batch in loader:
            with autocast(enabled=USE_AMP):
                p = model(batch['input_ids'].to(DEVICE),
                           batch['attention_mask'].to(DEVICE))
            preds.extend(p.cpu().float().numpy())
            trues.extend(batch['label'].numpy())
    return np.array(preds), np.array(trues)

def train_epochs(model, train_loader, val_loader, epochs=15, lr=2e-5,
                 patience=4, checkpoint_path=None):
    """Standard training loop with early stopping and AMP."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler    = GradScaler(enabled=USE_AMP)
    criterion = nn.MSELoss()
    best_val  = float('inf')
    counter   = 0

    for epoch in range(epochs):
        model.train()
        losses = []
        for batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            with autocast(enabled=USE_AMP):
                preds = model(batch['input_ids'].to(DEVICE),
                               batch['attention_mask'].to(DEVICE))
                loss  = criterion(preds, batch['label'].to(DEVICE))
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            losses.append(loss.item())
        scheduler.step()

        val_pred, val_true = evaluate_model(model, val_loader)
        val_rmse = np.sqrt(np.mean((val_pred - val_true)**2))
        print(f'  Epoch {epoch+1:>2} | loss={np.mean(losses):.4f} | val_rmse={val_rmse:.4f}')

        if val_rmse < best_val:
            best_val = val_rmse
            counter  = 0
            if checkpoint_path:
                torch.save(model.state_dict(), checkpoint_path)
        else:
            counter += 1
            if counter >= patience:
                print(f'  Early stop at epoch {epoch+1}')
                break

    if checkpoint_path:
        model.load_state_dict(torch.load(checkpoint_path))
    return model

train_loader, val_loader, test_loader = make_loaders(batch_size=32)
print('Loaders ready.')

## OPT 1 — Gradient Checkpointing
Recomputes activations during backward pass instead of storing them.
**Result:** ~30-40% VRAM reduction, ~20% slower training.

In [ ]:
from codecarbon import EmissionsTracker

print('OPT 1: Gradient Checkpointing')
print('=' * 45)

torch.cuda.empty_cache()

model_gc = ChemBERTaRegressor(MODEL_NAME).to(DEVICE)

# Enable gradient checkpointing on BERT encoder layers
model_gc.bert.gradient_checkpointing_enable()
print('  Gradient checkpointing: ENABLED')

res_tracker = ResourceTracker('opt_grad_checkpoint', './results/')
co2_tracker = EmissionsTracker(project_name='opt_gc', country_iso_code='BGR',
                                output_dir='./results/', log_level='error',
                                output_file='emissions_opt_gc.csv')

res_tracker.start()
co2_tracker.start()

model_gc = train_epochs(model_gc, train_loader, val_loader,
                         epochs=15, checkpoint_path='./models/opt_gc_best.pt')

kg_co2 = co2_tracker.stop()
res_gc = res_tracker.stop()

test_pred, test_true = evaluate_model(model_gc, test_loader)
metrics_gc = compute_metrics(test_true, test_pred)

print('\nResults vs Baseline:')
print(f'  R²:        {BASELINE["test_r2"]:.4f} → {metrics_gc["r2"]:.4f}')
print(f'  VRAM:      {BASELINE["peak_vram_mb"]:.0f} MB → {res_gc["peak_vram_mb"]:.0f} MB')
print(f'  Train:     {BASELINE["train_time_min"]:.1f} min → {res_gc["train_time_min"]:.1f} min')

vram_saved = BASELINE['peak_vram_mb'] - res_gc['peak_vram_mb']
print(f'  VRAM saved: {vram_saved:.0f} MB ({vram_saved/BASELINE["peak_vram_mb"]*100:.1f}%)')

torch.cuda.empty_cache()

## OPT 2 — Knowledge Distillation
Train a smaller student model to mimic the teacher's predictions.
**Result:** 3-5x smaller model, minimal accuracy loss.

In [ ]:
print('OPT 2: Knowledge Distillation')
print('=' * 45)

# Load the baseline teacher
teacher = ChemBERTaRegressor(MODEL_NAME).to(DEVICE)
teacher.load_state_dict(torch.load('./models/chemberta2_best.pt'))
teacher.eval()
print('  Teacher loaded (frozen)')

# Student: ChemBERTa with much smaller regression head
class StudentRegressor(nn.Module):
    """Smaller regression head — ~3x fewer parameters than teacher head."""
    def __init__(self, model_name):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        # Freeze most BERT layers — only fine-tune last 2
        for param in self.bert.parameters():
            param.requires_grad = False
        for layer in self.bert.encoder.layer[-2:]:
            for param in layer.parameters():
                param.requires_grad = True

        hidden = self.bert.config.hidden_size
        self.regressor = nn.Sequential(
            nn.Linear(hidden, 64),   # Much smaller than teacher's 256
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1)
        )

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.regressor(cls).squeeze(-1)

def distillation_loss(student_preds, teacher_preds, true_labels,
                       alpha=0.5, temperature=3.0):
    """
    Combined loss: alpha * MSE(student, true) + (1-alpha) * MSE(student, teacher)
    Temperature softens teacher predictions for better knowledge transfer.
    """
    hard_loss = nn.functional.mse_loss(student_preds, true_labels)
    soft_loss = nn.functional.mse_loss(
        student_preds / temperature,
        teacher_preds.detach() / temperature
    ) * (temperature ** 2)
    return alpha * hard_loss + (1 - alpha) * soft_loss

torch.cuda.empty_cache()
student = StudentRegressor(MODEL_NAME).to(DEVICE)

# Count parameters
teacher_params = sum(p.numel() for p in teacher.parameters())
student_trainable = sum(p.numel() for p in student.parameters() if p.requires_grad)
print(f'  Teacher params (total):    {teacher_params:,}')
print(f'  Student trainable params:  {student_trainable:,}')
print(f'  Compression ratio:         {teacher_params / student_trainable:.1f}x fewer updates')

res_tracker = ResourceTracker('opt_distillation', './results/')
co2_tracker = EmissionsTracker(project_name='opt_distill', country_iso_code='BGR',
                                output_dir='./results/', log_level='error',
                                output_file='emissions_opt_distill.csv')

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, student.parameters()),
    lr=1e-4, weight_decay=0.01
)
scaler = GradScaler(enabled=USE_AMP)

res_tracker.start()
co2_tracker.start()

best_val = float('inf')
patience_counter = 0

for epoch in range(20):
    student.train()
    losses = []
    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        lbls = batch['label'].to(DEVICE)

        with autocast(enabled=USE_AMP):
            student_out = student(ids, mask)
            with torch.no_grad():
                teacher_out = teacher(ids, mask)
            loss = distillation_loss(student_out, teacher_out, lbls)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        losses.append(loss.item())

    val_pred, val_true = evaluate_model(student, val_loader)
    val_rmse = np.sqrt(np.mean((val_pred - val_true)**2))
    print(f'  Epoch {epoch+1:>2} | loss={np.mean(losses):.4f} | val_rmse={val_rmse:.4f}')

    if val_rmse < best_val:
        best_val = val_rmse
        patience_counter = 0
        torch.save(student.state_dict(), './models/opt_student_best.pt')
    else:
        patience_counter += 1
        if patience_counter >= 4:
            print(f'  Early stop at epoch {epoch+1}')
            break

kg_co2 = co2_tracker.stop()
res_distill = res_tracker.stop()

student.load_state_dict(torch.load('./models/opt_student_best.pt'))
test_pred, test_true = evaluate_model(student, test_loader)
metrics_distill = compute_metrics(test_true, test_pred)

print('\nResults vs Baseline:')
print(f'  R²:        {BASELINE["test_r2"]:.4f} → {metrics_distill["r2"]:.4f}')
print(f'  VRAM:      {BASELINE["peak_vram_mb"]:.0f} MB → {res_distill["peak_vram_mb"]:.0f} MB')
print(f'  Train:     {BASELINE["train_time_min"]:.1f} min → {res_distill["train_time_min"]:.1f} min')

torch.cuda.empty_cache()

## OPT 3 — Optuna Hyperparameter Optimization
Find the smallest architecture that still hits our R² target.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

print('OPT 3: Optuna Hyperparameter Search')
print('=' * 45)
print('Goal: find smallest model config hitting R² > 0.70')
print()

R2_TARGET = 0.70  # Minimum acceptable accuracy

def objective(trial):
    torch.cuda.empty_cache()

    # Search space — architecture size
    hidden_size = trial.suggest_categorical('hidden_size', [64, 128, 256])
    dropout     = trial.suggest_float('dropout', 0.1, 0.4)
    lr          = trial.suggest_float('lr', 1e-5, 5e-5, log=True)
    batch_size  = trial.suggest_categorical('batch_size', [16, 32, 48])
    frozen_layers = trial.suggest_int('frozen_layers', 6, 11)  # How many BERT layers to freeze

    # Build model
    model = ChemBERTaRegressor(MODEL_NAME, hidden_size=hidden_size, dropout=dropout).to(DEVICE)

    # Freeze first N encoder layers
    for i, layer in enumerate(model.bert.encoder.layer):
        if i < frozen_layers:
            for param in layer.parameters():
                param.requires_grad = False

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    trial.set_user_attr('trainable_params', trainable)

    # Short training (3 epochs for HPO speed)
    t_loader, v_loader, _ = make_loaders(batch_size=batch_size)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=0.01
    )
    scaler    = GradScaler(enabled=USE_AMP)
    criterion = nn.MSELoss()

    for epoch in range(5):
        model.train()
        for batch in t_loader:
            optimizer.zero_grad(set_to_none=True)
            with autocast(enabled=USE_AMP):
                preds = model(batch['input_ids'].to(DEVICE),
                               batch['attention_mask'].to(DEVICE))
                loss  = criterion(preds, batch['label'].to(DEVICE))
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()

    val_pred, val_true = evaluate_model(model, v_loader)
    val_r2 = compute_metrics(val_true, val_pred)['r2']

    del model
    torch.cuda.empty_cache()

    # Penalize models that don't hit R² target
    if val_r2 < R2_TARGET:
        return float('inf')

    # Minimize trainable params (smaller = better)
    return trainable

study = optuna.create_study(direction='minimize',
                             sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=20, timeout=1800)  # max 30 min

best = study.best_params
print('\nBest config found:')
for k, v in best.items():
    print(f'  {k}: {v}')
print(f'  trainable_params: {study.best_trial.user_attrs.get("trainable_params", "N/A"):,}')

# Save best config
with open('./results/optuna_best.json', 'w') as f:
    json.dump(best, f, indent=2)
print('\nSaved to results/optuna_best.json')

## OPT 4 — INT8 Quantization (Inference)
Reduces model memory by 4x at inference time with minimal accuracy loss.

In [ ]:
import torch.quantization as tq

print('OPT 4: INT8 Dynamic Quantization (Inference)')
print('=' * 45)

# Load best baseline model for quantization
model_fp32 = ChemBERTaRegressor(MODEL_NAME).to('cpu')  # Quantization runs on CPU
model_fp32.load_state_dict(torch.load('./models/chemberta2_best.pt', map_location='cpu'))
model_fp32.eval()

# Apply dynamic INT8 quantization to linear layers
model_int8 = tq.quantize_dynamic(
    model_fp32,
    {nn.Linear},   # Only quantize Linear layers
    dtype=torch.qint8
)

# Compare model sizes
import os
torch.save(model_fp32.state_dict(), '/tmp/fp32_size_check.pt')
torch.save(model_int8.state_dict(), '/tmp/int8_size_check.pt')

fp32_size = os.path.getsize('/tmp/fp32_size_check.pt') / 1e6
int8_size = os.path.getsize('/tmp/int8_size_check.pt') / 1e6

print(f'  FP32 model size: {fp32_size:.1f} MB')
print(f'  INT8 model size: {int8_size:.1f} MB')
print(f'  Compression:     {fp32_size/int8_size:.1f}x smaller')

# Measure inference speed
sample_batch = next(iter(val_loader))
ids  = sample_batch['input_ids']
mask = sample_batch['attention_mask']

# FP32 speed
model_fp32.eval()
t0 = time.time()
for _ in range(10):
    with torch.no_grad():
        _ = model_fp32(ids, mask)
fp32_ms = (time.time() - t0) / 10 * 1000

# INT8 speed
model_int8.eval()
t0 = time.time()
for _ in range(10):
    with torch.no_grad():
        _ = model_int8(ids, mask)
int8_ms = (time.time() - t0) / 10 * 1000

print(f'\nInference speed (CPU, batch=32):')
print(f'  FP32: {fp32_ms:.1f} ms/batch')
print(f'  INT8: {int8_ms:.1f} ms/batch')
print(f'  Speedup: {fp32_ms/int8_ms:.2f}x')

# Accuracy check on test set (CPU loaders)
cpu_loader = DataLoader(
    SMILESDataset(df_test['Drug'].tolist(), df_test['Y_log'].values, tokenizer),
    batch_size=32, shuffle=False
)

preds_fp32, trues = [], []
preds_int8 = []

for batch in cpu_loader:
    with torch.no_grad():
        p32 = model_fp32(batch['input_ids'], batch['attention_mask'])
        p8  = model_int8(batch['input_ids'], batch['attention_mask'])
    preds_fp32.extend(p32.numpy())
    preds_int8.extend(p8.numpy())
    trues.extend(batch['label'].numpy())

m_fp32 = compute_metrics(trues, preds_fp32)
m_int8 = compute_metrics(trues, preds_int8)

print(f'\nAccuracy comparison:')
print(f'  FP32 R²: {m_fp32["r2"]:.4f}')
print(f'  INT8 R²: {m_int8["r2"]:.4f}')
print(f'  R² drop: {m_fp32["r2"] - m_int8["r2"]:.4f}  (should be < 0.01)')

torch.save(model_int8.state_dict(), './models/opt_int8.pt')
print('\nINT8 model saved to models/opt_int8.pt')

## Final Comparison — All Optimizations vs Baseline

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Collect all optimization results
opt_results = {
    'Baseline':              {'r2': BASELINE['test_r2'],       'vram': BASELINE['peak_vram_mb'], 'time': BASELINE['train_time_min']},
    'Grad Checkpoint':       {'r2': metrics_gc['r2'],          'vram': res_gc['peak_vram_mb'],    'time': res_gc['train_time_min']},
    'Distillation':          {'r2': metrics_distill['r2'],     'vram': res_distill['peak_vram_mb'], 'time': res_distill['train_time_min']},
    'INT8 (inference)':      {'r2': m_int8['r2'],              'vram': int8_size*10,              'time': BASELINE['train_time_min']},
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Optimization Results vs Baseline', fontsize=13, fontweight='bold')

names  = list(opt_results.keys())
colors = ['#888', '#5C85D6', '#59B87A', '#E8874A', '#C85BAF']

# R²
ax = axes[0]
r2s = [opt_results[n]['r2'] for n in names]
bars = ax.bar(names, r2s, color=colors[:len(names)], edgecolor='white')
ax.set_title('Test R²')
ax.set_ylim(0, 1)
ax.axhline(y=BASELINE['test_r2'], color='red', linestyle='--', alpha=0.5, label='Baseline')
ax.legend()
for bar, val in zip(bars, r2s):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', fontsize=8)
ax.tick_params(axis='x', rotation=20)

# VRAM
ax = axes[1]
vrams = [opt_results[n]['vram'] for n in names]
bars = ax.bar(names, vrams, color=colors[:len(names)], edgecolor='white')
ax.set_title('Peak VRAM (MB) ↓ lower better')
for bar, val in zip(bars, vrams):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f'{val:.0f}', ha='center', fontsize=8)
ax.tick_params(axis='x', rotation=20)

# Train time
ax = axes[2]
times = [opt_results[n]['time'] for n in names]
bars = ax.bar(names, times, color=colors[:len(names)], edgecolor='white')
ax.set_title('Train Time (min) ↓ lower better')
for bar, val in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{val:.1f}m', ha='center', fontsize=8)
ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('./results/04_optimization_results.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nSummary saved to results/04_optimization_results.png')
print('\n✓ Ready for 05_generate.ipynb')